# Apache Spark

## Instalando o ambiente

O jeito mais simples de começar a trabalhar com Spark é instalar um container com tudo pronto! No site https://hub.docker.com/r/jupyter/pyspark-notebook vemos uma imagem Docker que já vem com `pyspark` e `jupyter lab`. Instale a imagem com o comando:

```bash
docker pull jupyter/pyspark-notebook
```

Vamos iniciar o ambiente de trabalho com o comando `docker run`. Para isso precisamos tomar alguns cuidados:

1) Temos que mapear nosso diretorio local de trabalho para um diretório interno do container, de modo que alterações feitas dentro do container (nesta pasta escolhida) sejam gravadas no nosso diretorio local. No container temos um usuário padrão com *username* `jovyan`. No *homedir* desse usuario temos uma pasta vazia `work`, que vai servir como local de mapeamento do nosso diretorio local de trabalho. Podemos então fazer esse mapeamendo com a opção `-v` do comando `docker run` da seguinte forma:

```bash
-v <diretorio>:/home/jovyan/work
```

onde `<diretorio>` representa seu diretorio local de trabalho.

2) Para acessar o `jupyter notebook` e o *dashboard* do Spark a partir do nosso *browser* favorito temos que abrir algumas portas do container com a opção `-p`. As portas são `8888` (para o próprio `jupyter notebook`) e `4040` (para o *dashboard* do Spark). Ou seja, adicionaremos às opções do `docker run`o seguinte:

```bash
-p 8888:8888 -p 4040:4040
```

Desta forma, ao acessar `localhost:8888` na nossa máquina, estaremos acessando o servidor Jupyter na porta 8888 interna do container.

3) Vamos iniciar o container no modo interativo, e vamos especificar que o container deve ser encerrado ao fechar o servidor Jupyter. Faremos isso com as opções `-it` e `-rm`

Antes de executar, garanta que as portas 4040 e 8888 estão livres (sem jupyter já executando) ou altere o comando. Ainda, esteja na pasta da aula ao executar, assim apenas ela será exposta ao container.

Portanto, o comando completo que eu uso na minha máquina Linux para iniciar o container é:

```bash
docker run \
    -it \
    --rm \
    -p 8888:8888 \
    -p 4040:4040 \
    -v "`pwd`":/home/jovyan/work \
    jupyter/pyspark-notebook


```


Se estiver no Windows estes comandos, utilize:

- No Powershell: `docker run -it --rm -p 8888:8888 -p 4040:4040 -v ${PWD}:/home/jovyan/work jupyter/pyspark-notebook`

- No Prompt de comando: `docker run -it --rm -p 8888:8888 -p 4040:4040 -v %cd%:/home/jovyan/work jupyter/pyspark-notebook`


Para facilitar a vida eu coloco esse comando em um arquivo `inicia.sh`. Engenheiros, façam do jeito que preferirem!

Agora abra esse notebook lá no container!


## Iniciando o Spark

Vamos iniciar o ambiente Spark. Para isso vamos:

1) Criar um objeto de configuração do ambiente Spark. Nossa configuração será simples: vamos especificar que o nome da nossa aplicação Spark é "Minha aplicação", e que o *master node* é a máquina local, usando todos os *cores* disponíveis. Aplicações reais de Spark são configuradas de modo ligeiramente diferente: ao especificar o *master node* passamos uma URL real, com o endereço do nó gerente do *cluster* Spark.

2) Vamos criar um objeto do tipo `SparkContext` com essa configuração

In [ ]:
import os
from pyspark.sql import SparkSession

base_path = os.path.join(os.getcwd(), "spark")
warehouse_dir = os.path.join(base_path, "spark-warehouse")
metastore_db = os.path.join(base_path, "metastore_db")

spark = SparkSession.builder \
    .appName("MinhaAplicacao") \
    .config("spark.sql.warehouse.dir", warehouse_dir) \
    .config("javax.jdo.option.ConnectionURL", f"jdbc:derby:;databaseName={metastore_db};create=True") \
    .config("hive.metastore.schema.verification", "false") \
    .config("hive.metastore.datanucleus.fixedDatastore", "false") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Warehouse dir: {warehouse_dir}")
print(f"Metastore DB: {metastore_db}")

A partir deste momento você pode monitorar seus *jobs* Spark em http://localhost:4040

## Spark: SQL

Nas últimas aulas de Spark, aprendemos sobre RDD (Resilient Distributed Datasets), que é a estrutura de dados fundamental do Spark. Com o RDD, podemos executar operações paralelas e distribuídas em grandes conjuntos de dados em um cluster.

<img src="cluster-overview.png">

Fonte: https://spark.apache.org/docs/latest/img/cluster-overview.png

Aprendemos que os RDDs são imutáveis e podem ser criados a partir de dados armazenados em arquivos ou gerados por transformações de outros RDDs. Além disso, vimos como aplicar diferentes tipos de transformações, como `map`, `filter` e `reduce` para manipular nossos dados.

Agora, vamos aprender sobre a interface de DataFrames do PySpark. Os DataFrames são uma abstração de alto nível construída em cima dos RDDs, que permitem a manipulação de dados estruturados de forma mais eficiente. Eles fornecem uma API mais fácil e intuitiva para trabalhar com dados tabulares, além de permitir a execução de consultas SQL diretamente nos dados.

Sim, você não leu errado, conseguiremos utilizar SQL! Em nossas aulas, discutimos como geralmente servidores de banco de dados são otimizados para IO (leitura e escrita de dados) e não para processamento. Com o uso do Spark e a interface SQL, conseguiremos aplicar nossos conhecimentos em um ambiente otimizado para processamento de dados em larga escala enquanto utilizamos uma interface mais amigável, permitindo que as tarefas de análise sejam realizadas de forma mais rápida e eficiente.

Com os DataFrames, podemos executar tarefas como filtragem, agregação, junção e ordenação dos dados com muito mais facilidade. Além disso, eles oferecem suporte a diversos formatos de arquivos, incluindo CSV, Parquet, JSON e avro.

**Obs**: apesar de serem DataFrames, não são Pandas DataFrames! Apesar de podermos transformar estes DataFrames em Pandas, esta é uma operação que deve ser evitada ao máximo, pois assim perdemos a característica distribuída do Spark!

## Base de Dados

Iremos utilizar a base de dados **SF Bay Area Bike Share** do [Kaggle](https://www.kaggle.com/datasets/benhamner/sf-bay-area-bike-share?resource=download). Para fazer o download, acesse https://www.kaggle.com/datasets/benhamner/sf-bay-area-bike-share?resource=download

Provavelmente o download pelo Kaggle irá demorar. Enquanto ele não finalize, utilize como alternativa o zip disponível no Blackboard. Ele possui os mesmos CSVs, exceto um gigantesco (`status.csv`)!

## Primeiro exemplo

Vamos abrir o arquivo `station.csv`. Deixe ele em uma pasta `data` dentro da pasta da aula.

In [ ]:
df_station = spark.read.csv("data/station.csv", header=True, inferSchema=True)

Vamos exibir algumas linhas

In [ ]:
df_station.show(5)

Uma maneira alternativa de exibir de forma mais bonita! **Cuidado**: transformações para Pandas em geral devem ser evitadas!

In [ ]:
df_station.limit(3).toPandas()

Para ver o *schema* do DataFrame, utilize:

In [ ]:
df_station.printSchema()

Podemos utilizar `count` e `columns` para descobrir o *shape* do DataFrame:

In [ ]:
df_station.count()

In [ ]:
df_station.columns

In [ ]:
print(f"{df_station.count()} linhas e {len(df_station.columns)} colunas")

Vamos encontrar todas as cidades diferentes nas quais temos estações em nossa base de dados

In [ ]:
from pyspark.sql.functions import col

distinct_cities = df_station.select(col("city")).distinct()

distinct_cities.show()

Acesse a documentação do `select` em 
https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.sql.DataFrame.select.html
e do `col` em https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.col.html

## Criar Databases

Vamos tentar trabalhar de forma mais semelhante ao praticado no MySQL!

Será que conseguimos criar nossas próprias databases e tabelas?!

Sim! Para criar uma database `bike`, podemos fazer:

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bike")

## Criar Tabelas

Em um ambiente de processamento em larga escala como o Spark, as tabelas podem surgir tanto em um processo de ingestão, quando arquivos externos a base são ingeridos para disponibilizar dados aos cientistas ou analistas de dados, quanto quando tabelas são reprocessadas para gerar visões necessárias para satisfazer algum projeto (Ex: um projeto Machine Learning para predição de custos).

Para exportar um DataFrame como tabela, podemos fazer

In [ ]:
df_station.write.mode("overwrite").saveAsTable("bike.station")

Para criar uma view, utilize

In [ ]:
df_station.createOrReplaceTempView("view_station")

Vá até seu navegador de arquivos (Windows Explorer, Nautilius). Verifique que foi criada a pasta `spark-warehouse`. Acesse esta pasta e confira seu conteúdo!

Perceba que nossas tabelas foram salvas em arquivos `.parquet`. O Apache Parquet é um formato de arquivo open-source projetado para armazenar dados em colunas. Ele é otimizado para consulta e processamento eficiente de grandes quantidades de dados estruturados e semi-estruturados, especialmente em ambientes de big data. Ao contrário de outros formatos de arquivo que armazenam dados em linhas, o Parquet armazena dados em colunas, o que permite uma compressão mais eficiente e um desempenho de consulta mais rápido.

<img src="parquet.gif">
Fonte: https://parquet.apache.org/images/FileLayout.gif

O Parquet foi projetado para trabalhar bem com frameworks distribuídos como Hadoop, Spark e Hive, permitindo que os usuários consultem e processem dados em escala. Ele também suporta tipos de dados complexos, como arrays e estruturas aninhadas, o que o torna flexível o suficiente para lidar com uma ampla variedade de casos de uso. Em resumo, o Apache Parquet é uma ferramenta útil para gerenciar e processar grandes volumes de dados de forma eficiente e escalável.

Veja mais em https://parquet.apache.org/

Também podemos especificar partições na criação das tabelas. Considere um caso em que a tabela de `station` será dividida pelas cidades

In [ ]:
df_station.write.partitionBy("city").mode("overwrite").saveAsTable("bike.station")

Volte ao navegador de arquivos e perceba as alterações. Deve ter sido criada uma pasta por cidade.

## SQL Query

Pronto, agora podemos utilizar queries como fizemos no MySQL!

In [ ]:
minha_query = """
SELECT *
  FROM bike.station
 LIMIT 2
 """

df_exemplo = spark.sql(minha_query)
df_exemplo.show(5)

Utilizando a view...

In [ ]:
minha_query = """
SELECT s.name,
       s.city,
       s.dock_count
  FROM view_station s
 LIMIT 3
 """

df_exemplo = spark.sql(minha_query)
df_exemplo.show(5)

**Atividade 1**: Crie na base `bike` uma tabela `weather` a partir do arquivo `weather.csv`. A coluna `date` deve ser convertida para o tipo `date` (use `to_date`).

In [ ]:
# Seu código AQUI!

**Atividade 2**: Crie na base `bike` uma tabela `trip` a partir do arquivo `trip.csv`

In [ ]:
# Seu código AQUI!

**Atividade 3**: Crie na base `bike` uma tabela `status` a partir do arquivo `status.csv`

In [ ]:
# Seu código AQUI!

**Atividade 4**: Conte a quantidade de linhas em cada tabela.

In [ ]:
# Seu código AQUI!

**Atividade 5**: Conte a quantidade de corridas (`trip`) com cada estação como **origem**.

In [ ]:
# Seu código AQUI!

**Atividade 6**: Conte a quantidade de corridas (`trip`) com cada estação como **destino**.

In [ ]:
# Seu código AQUI!

### Joins

Podemos utilizar **join** e os demais recursos (funções de agregação, agrupamentos...) que utilizávamos no MySQL. Veja um exemplo onde retornaremos algumas informações da estação fazendo um **INNER JOIN** e retornando as informações de `status` da estação.

In [ ]:
minha_query = """
SELECT s.name,
       s.city,
       s.dock_count,
       t.*
  FROM bike.station s,
       bike.status t
WHERE t.station_id = s.id
 LIMIT 3
 """

df_info_st = spark.sql(minha_query)
df_info_st.show(8)

Agora com o `INNER JOIN` explícito, ao invés de fazer pelo `WHERE`!

In [ ]:
minha_query = """
SELECT s.name,
       s.city,
       s.dock_count,
       t.*
  FROM bike.station s
       INNER JOIN bike.status t ON station_id = s.id
 LIMIT 3
 """

df_info_st = spark.sql(minha_query)
df_info_st.show(8)

Você irá perceber uma certa demora para retorno dos resultados. Estamos em um ambiente que propicia processamento em larga escala de forma distribuída. Conseguimos recuperar, agrupar, resumir e até treinar modelos de Machine Learning, mas isto virá com um custo!

**Atividade 7**: Crie um DataFrame a partir de uma **Query SQL** que retorne o `id`, `name`, `city` além da quantidade média de bicicletas disponíveis nos `status` de cada estação.

Dicas: junção e agrupamento!

In [ ]:
# Seu código AQUI!

### Resolvendo com API DataFrame

Vamos ver como resolver o Exercício 7 utilizando a API de DataFrames.

Inicialmente, iremos ler a tabela de status. Já temos um DataFrame para esta tabela, entretanto, iremos criar outro para fins ditáticos! Perceba que desta vez iremos fazer o processo inverso e ler a partir da tabela! 

In [ ]:
df_status1 = spark.read.table("bike.status")
df_status1.show(2)

Então, lemos a tabela de estações em um DataFrame

In [ ]:
df_station1 = spark.read.table("bike.station")
df_station1.show(2)

Para fazer o join, utilizamos:

In [ ]:
df_join = df_station1.join(df_status1, col("id") == col("station_id"), "inner")
df_join.show(2)

E utilizamos `avg` como função de agregação para obter os resultados desejados:

In [ ]:
from pyspark.sql.functions import avg

df_join.groupBy("id", "name", "city").agg(avg("bikes_available")).show(10)

Perceba que os resultados serão os mesmos da opção resolvida com SQL.

Em geral, a escolha pela interface de SQL ou DataFrame em um ambiente de trabalho com Spark levará em consideração questões como padronização e a familiaridade dos desenvolvedores com uma interface ou outra. Seja qual a escolha, o tempo de processamento também deverá ser muito parecido, uma vez que o Spark realiza uma "tradução" no momento de execução.

### DataFrames: Principais funções

Nas seções anteriores utilizamos SQL para consultar os dados. Agora vamos praticar as mesmas operações usando diretamente a **API de DataFrames do PySpark**. Essa API é mais programática e se integra bem com Python, além de ser equivalente em desempenho ao SQL (o Spark otimiza as duas da mesma forma).

A tabela abaixo resume a correspondência entre comandos SQL e os métodos da API:

| SQL | DataFrame API |
|-----|--------------|
| `WHERE condição` | `.filter(col("col") > valor)` ou `.where(...)` |
| `SELECT DISTINCT col` | `.distinct()` / `.dropDuplicates(["col"])` |
| `SELECT col, col*2 AS novo` | `.select(...)` + `.withColumn(...)` |
| `WHERE col IS NOT NULL` | `.dropna(subset=["col"])` |
| `IFNULL(col, val)` | `.fillna({"col": val})` |
| `CAST(col AS tipo)` | `.withColumn("c", col("c").cast("tipo"))` |
| `col AS novo_nome` *(renomear)* | `.withColumnRenamed("old", "new")` |
| `GROUP BY ... COUNT(*), AVG(col)` | `.groupBy(...).agg(count(), avg(...))` |
| `ORDER BY col DESC` | `.orderBy(col("col").desc())` |
| `JOIN t2 ON t1.k = t2.k` | `.join(df2, df1["k"] == df2["k"], "inner")` |
| `UNION ALL` | `.union(df2)` / `.unionByName(df2)` |
| `CASE WHEN c THEN v ELSE w END` | `when(condição, valor).otherwise(valor)` |

Documentação: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html

Primeiro, importamos algumas funções:

In [ ]:
# Estes imports também foram realizados no servidor de testes
import pyspark.sql.functions as func
from pyspark.sql.functions import col, count, avg, min, max, sum, round, when

Então vamos recarregar os DataFrames das tabelas já criadas:

In [ ]:
df_station = spark.read.table("bike.station")
df_trip = spark.read.table("bike.trip")
df_weather = spark.read.table("bike.weather")
df_status = spark.read.table("bike.status")

Vamos agora configurar o insperautograder para que funcione dentro do container do Docker. Primeiro, instale e importe a biblioteca:

In [ ]:
# Descomente para instalar
# !pip install git+https://github.com/macielcalebe/insperautograding.git

In [ ]:
import insperautograder.jupyter as ia
from dotenv import load_dotenv

Se inicializou o container na pasta de aulas, você já deve ter acesso ao arquivo `.env` que contém as variáveis de ambiente necessárias para o autograder funcionar. Agora, vamos carregar essas variáveis de ambiente:

In [ ]:
load_dotenv(override=True)
ia.tasks()

In [ ]:
ia.grades(task="spark_dataframes")

In [ ]:
ia.grades(by="TASK", task="spark_dataframes")

In [ ]:
# Calcular média de ATVs, desconsiderando duas atividades (divide por n-2)
ia.average(excluded_count=2)

---
### `filter` / `where`

Filtra linhas com base em uma condição. Equivalente ao `WHERE` do SQL.

```python
df.filter(col("coluna") > valor)
df.where(col("coluna") == "texto")   # sintaxe alternativa, mesma função
```

Vamos filtrar viagens com mais de 1 minuto de duração (60 segundos):

In [ ]:
df_trip.filter(col("duration") > 60).show(5)

`filter` e `where` são equivalentes. Veja uma sintaxe alternativa:

In [ ]:
df_trip.where(col("subscription_type") == "Subscriber").show(5)

Podemos utilizar múltiplas condições com `&` (AND) ou `|` (OR):

In [ ]:
df_trip.filter((col("duration") > 60) & (col("subscription_type") == "Subscriber")).show(5)

**Exercício 1**: Crie uma função `ex01` que recebe `df_station` e retorna um DataFrame contendo apenas as estações com `dock_count` entre 10 e 12 (inclusive).

In [ ]:
def ex01(df_station):
    # Seu código AQUI!
    return df_station  # Pode alterar o retorno, se necessário!

ex01(df_station).show(30)

In [ ]:
ia.sender(answer="ex01", task="spark_dataframes", question="ex01", answer_type="pycode")

### `select`

É equivalente ao `SELECT` do SQL. Aceita nomes de colunas, expressões e aliases.

```python
df.select("col1", "col2")
df.select(col("col1"), (col("col2") * 2).alias("col2_dobro"))
```

Selecionar colunas específicas:

In [ ]:
df_trip.select("id", "start_station_name", "end_station_name", "duration").show(5)

Usar expressões com `alias` dentro do `select`:

In [ ]:
df_trip.select(
    "id",
    "start_station_name",
    (col("duration") / 60).alias("duration_min")
).show(5)

### `orderBy`

Ordena o DataFrame — equivalente ao `ORDER BY`. Use `.desc()` para ordem decrescente.

```python
df.orderBy(col("coluna").asc())    # crescente (padrão)
df.orderBy(col("coluna").desc())   # decrescente
df.orderBy("colA", col("colB").desc())  # múltiplas colunas
```

As 5 viagens mais longas, ordenando por duração decrescente:

In [ ]:
df_trip.orderBy(col("duration").desc()).select("id", "start_station_name", "duration").show(5)

Ordenar por múltiplas colunas, sendo cidade crescente, depois docks decrescente:

In [ ]:
df_station.orderBy("city", col("dock_count").desc()).show(10)

**Exercício 2**: Crie uma função `ex02` que recebe `df_station` e retorna um DataFrame com o `name`e `id` de todas as estações da cidade de San Jose, ordenadas por `dock_count` de forma decrescente.

In [ ]:
def ex02(df_station):
    # Seu código AQUI!
    return df_station  # Pode alterar o retorno, se necessário!

ex02(df_station).toPandas()

In [ ]:
ia.sender(answer="ex02", task="spark_dataframes", question="ex02", answer_type="pycode")

### `distinct` / `dropDuplicates`

Remove linhas duplicadas — equivalente ao `SELECT DISTINCT` do SQL.

```python
df.distinct()                         # todas as colunas devem ser iguais
df.dropDuplicates(["col1", "col2"])   # considera apenas as colunas indicadas
```

> `dropDuplicates` é mais flexível: permite remover duplicatas com base em um subconjunto de colunas, mantendo a primeira ocorrência.

Cidades distintas das estações:

In [ ]:
df_station.select("city").distinct().show()

Tipos de assinatura distintos em viagens:

In [ ]:
df_trip.select("subscription_type").distinct().show()

`dropDuplicates`: manter apenas a primeira estação por cidade (ignora demais colunas):

In [ ]:
df_station.dropDuplicates(["city"]).select("city", "name").show()

In [ ]:
df_weather.show(10)

**Exercício 3**: Crie uma função `ex03` que recebe o DataFrame `df_weather` e retorna um DataFrame com os zip codes distintos que tiveram temperatura média maior que 75 graus fahrenheit. O resultado deve conter apenas uma coluna chamada `zip_code` e deve estar ordenado por esta coluna de forma crescente.

In [ ]:
def ex03(df_weather):
    # Seu código AQUI!
    return df_weather  # Pode alterar o retorno, se necessário!

ex03(df_weather).toPandas()

In [ ]:
ia.sender(answer="ex03", task="spark_dataframes", question="ex03", answer_type="pycode")

**Exercício 04**: Crie uma função `ex04` que recebe `df_trip` e retorna um DataFrame com os **pares distintos** de `(start_station_name, end_station_name)`, ordenados por `start_station_name` e depois por `end_station_name`.

**Obs**:
- Considere apenas viagens com duração de `10` **horas**, com tolerância de `10` **minutos** para mais ou para menos (intervalo fechado)
- Considere apenas tipos de assinatura `Customer`

In [ ]:
def ex04(df_trip):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex04(df_trip).show(30)

In [ ]:
ia.sender(answer="ex04", task="spark_dataframes", question="ex04", answer_type="pycode")

### `dropna` / `fillna`

Lida com valores nulos (`null`) — equivalente a `WHERE col IS NOT NULL` ou `IFNULL(col, val)` no SQL.

```python
df.dropna()                        # remove linhas com QUALQUER nulo
df.dropna(subset=["col1", "col2"]) # remove linhas com nulo em colunas específicas
df.dropna(how="all")               # remove apenas linhas onde TODAS as colunas são nulas

df.fillna(0)                          # substitui todos os nulos por 0 (numéricos)
df.fillna("desconhecido")             # substitui nulos em colunas string
df.fillna({"col1": 0, "col2": "N/A"}) # valores diferentes por coluna
```

Contar não nulos por coluna (técnica útil para exploração de dados):

Perceba que a coluna `zip_code` tem valores nulos.

In [ ]:
from pyspark.sql.functions import isnan, isnull

null_counts = df_trip.select([
    count(col(c)).alias(c) for c in df_trip.columns
])
total = df_trip.count()
print(f"Total de linhas: {total}")
null_counts.show()

Vamos fazer o mesmo para o DataFrame `df_weather`:

In [ ]:
from pyspark.sql.functions import isnan, isnull

null_counts = df_weather.select([
    count(col(c)).alias(c) for c in df_weather.columns
])
total = df_weather.count()
print(f"Total de linhas: {total}")
null_counts.show()

Remover linhas onde `zip_code` é nulo com `dropna`:

In [ ]:
df_trip.dropna(subset=["zip_code"]).show(5)

Substituir nulos com `fillna` — valores diferentes por coluna:

In [ ]:
# Comando deixado apenas como exemplo, pois a coluna `subscription_type` não tem valores nulos.
df_trip.fillna({"zip_code": "0000000", "subscription_type": "Unknown"}).show(5)

**Exercício 05**: Crie uma função `ex05` que recebe `df_weather` e um inteiro `k` e retorna um DataFrame com as colunas `date`, `zip_code`, `events`, `max_temperature_f` e `max_gust_speed_mph`, onde:

- linhas com `max_gust_speed_mph` nulo são **removidas**
- valores nulos em `events` são **substituídos** por `"Sem evento"`

**Obs**:
- Considere apenas os dados a partir de `2014`.
- Ordene por `date` e depois por `zip_code`, ambos de forma crescente
- Limite o retorno para `k` linhas.

In [ ]:
def ex05(df_weather, k):
    # Seu código AQUI!
    return df_weather  # Pode alterar o retorno, se necessário!

ex05(df_weather, 5).toPandas()

In [ ]:
ia.sender(answer="ex05", task="spark_dataframes", question="ex05", answer_type="pycode")

### `withColumn`

Adiciona uma nova coluna ou sobrescreve uma existente — equivalente ao `col AS alias` no `SELECT`.

```python
df.withColumn("nova_col", expressão)
df.withColumn("col_existente", col("col_existente").cast("double"))  # altera tipo
```

Adicionar uma coluna calculada de duração em horas:

In [ ]:
df_trip.withColumn("duration_hours", round(col("duration") / 3600.0, 2)).show(5)

Encadear `withColumn` para adicionar múltiplas colunas de uma vez:

In [ ]:
df_trip \
    .withColumn("duration_min", round(col("duration") / 60.0, 1)) \
    .withColumn("duration_hours", round(col("duration") / 3600.0, 2)) \
    .select("id", "duration", "duration_min", "duration_hours") \
    .show(5)

### `withColumnRenamed` / `drop` / `cast`

Completam a família de transformações de schema:

```python
# Renomear coluna
df.withColumnRenamed("nome_antigo", "nome_novo")

# Remover uma ou mais colunas
df.drop("col1", "col2")

# Converter tipo de dados (cast) — geralmente usado dentro de withColumn
df.withColumn("col", col("col").cast("double"))
df.withColumn("col", col("col").cast("integer"))
df.withColumn("col", col("col").cast("string"))
```

> **Tipos comuns**: `"string"`, `"integer"`, `"long"`, `"double"`, `"float"`, `"boolean"`, `"date"`, `"timestamp"`

Renomear colunas com `withColumnRenamed`:

In [ ]:
df_station \
    .withColumnRenamed("dock_count", "vagas") \
    .withColumnRenamed("name", "nome_estacao") \
    .show(5)

Remover colunas desnecessárias com `drop`:

In [ ]:
df_station.drop("lat", "long", "installation_date").show(5)

Converter tipos de dados com `cast`:

In [ ]:
df_station \
    .withColumn("id", col("id").cast("string")) \
    .withColumn("dock_count", col("dock_count").cast("double")) \
    .printSchema()

**Exercício 06**: Crie uma função `ex06` que recebe `df_station` e retorna um DataFrame com:
- as colunas `lat`, `long` e `installation_date` **removidas**
- a coluna `dock_count` **renomeada** para `vagas`
- a coluna `id` **convertida** para o tipo `string`

O resultado final deve ter as colunas `id` (string), `name`, `city` e `vagas`.

In [ ]:
def ex06(df_station):
    # Seu código AQUI!
    return df_station  # Pode alterar o retorno, se necessário!

ex06(df_station).show(5)
ex06(df_station).printSchema()

In [ ]:
ia.sender(answer="ex06", task="spark_dataframes", question="ex06", answer_type="pycode")

### `groupBy` + `agg`

Agrupa linhas e aplica funções de agregação, de forma equivalente ao `GROUP BY` com `COUNT`, `AVG`, etc.

```python
df.groupBy("col").agg(
    count("id").alias("total"),
    avg("valor").alias("media"),
    min("valor").alias("minimo"),
    max("valor").alias("maximo")
)
```

Total de viagens e duração média por tipo de assinatura:

In [ ]:
df_trip.groupBy("subscription_type").agg(
    count("id").alias("total_viagens"),
    round(avg("duration"), 2).alias("duracao_media_seg"),
    max("duration").alias("duracao_max_seg")
).show()

Agrupar por múltiplas colunas — quantidade de estações e média de docks por cidade:

In [ ]:
df_station.groupBy("city").agg(
    count("id").alias("qtd_estacoes"),
    round(avg("dock_count"), 1).alias("media_docks")
).show()

### `join`

Combina dois DataFrames com base em uma condição, de forma equivalente ao `JOIN` do SQL.

```python
df1.join(df2, condição, tipo)
# tipo: "inner" (padrão), "left", "right", "outer", "semi", "anti"
```

> **Atenção:** quando as duas tabelas têm colunas com o mesmo nome (ex: `id`), use `df["col"]` para evitar ambiguidade.

Veja um exemplo de **inner join** para enriquecer viagens com informações da estação de partida (use `df["col"]` para evitar ambiguidade de colunas com mesmo nome):

In [ ]:
df_trip.join(df_station, df_trip["start_station_id"] == df_station["id"], "inner") \
    .select(df_trip["id"], "start_station_name", "city", "duration") \
    .show(5)

Left join para retornar todas as viagens, mesmo sem estação correspondente:

In [ ]:
df_trip.join(df_station, df_trip["start_station_id"] == df_station["id"], "left") \
    .select(df_trip["id"], "city") \
    .show(5)

### `union` / `unionByName`

Empilha dois DataFrames com o mesmo schema, de forma equivalente ao `UNION ALL` do SQL.

```python
df1.union(df2)             # combina por posição das colunas (schema deve ser idêntico)
df1.unionByName(df2)       # combina pelo nome das colunas (mais seguro)
df1.unionByName(df2, allowMissingColumns=True)  # preenche com null colunas ausentes
```

> **Diferença de `join`**: `union` empilha linhas (vertical); `join` combina colunas (horizontal).  
> O `union` do Spark se comporta como `UNION ALL` do SQL — **não remove duplicatas**. Para remover, encadeie `.distinct()`.


Preparar dois DataFrames — estações de San Francisco e San Jose:

In [ ]:
df_sf = df_station.filter(col("city") == "San Francisco").select("id", "name", "city")
df_sj = df_station.filter(col("city") == "San Jose").select("id", "name", "city")

`union` empilha as linhas (como `UNION ALL` do SQL — **não** remove duplicatas):

In [ ]:
df_sf.union(df_sj).show(10)

`unionByName` — mais seguro, combina pelo nome das colunas:

In [ ]:
df_sf.unionByName(df_sj).count()

Para remover duplicatas, encadeie `.distinct()` após o `union`:

In [ ]:
df_sf.union(df_sj).distinct().show(5)

**Exercício 07**: Crie uma função `ex07` que recebe dois DataFrames e retorna um único DataFrame com as estações das cidades `"Redwood City"` e `"Palo Alto"` **combinadas**. O resultado deve ter as colunas `id`, `name`, `city` e `dock_count`, **sem duplicatas**, ordenado por `city` e depois por `name`.

In [ ]:
def ex07(df_rc, df_pa):
    # Seu código AQUI!
    return df_rc  # Pode alterar o retorno, se necessário!

# Estações de Redwood City e Palo Alto
df_rc = df_station.filter(col("city") == "Redwood City").select("id", "name", "city", "dock_count")
df_pa = df_station.filter(col("city") == "Palo Alto").select("id", "name", "city", "dock_count")

ex07(df_rc, df_pa).show()

In [ ]:
ia.sender(answer="ex07", task="spark_dataframes", question="ex07", answer_type="pycode")

### `when` / `otherwise`

Cria uma expressão condicional, de forma equivalente ao `CASE WHEN ... THEN ... ELSE ... END` do SQL. Normalmente usado dentro de `withColumn` ou `select`.

```python
from pyspark.sql.functions import when

when(condição, valor_se_true)
    .when(outra_condição, outro_valor)  # pode encadear
    .otherwise(valor_padrão)            # equivale ao ELSE
```

Classificar viagens em curta, média e longa com `when`/`otherwise`:

In [ ]:
df_trip.withColumn("tipo",
    when(col("duration") < 600,  "curta")
    .when(col("duration") < 3600, "média")
    .otherwise("longa")
).select("id", "duration", "tipo").show(10)

Criar uma flag binária — viagem circular (início = fim):

In [ ]:
df_trip.withColumn("is_round_trip",
    when(col("start_station_id") == col("end_station_id"), 1).otherwise(0)
).select("id", "start_station_id", "end_station_id", "is_round_trip").show(5)

### `write`

Persiste um DataFrame em disco ou no catálogo. O método `.write` retorna um `DataFrameWriter` com os seguintes formatos e modos principais:

| Método | Descrição |
|--------|-----------|
| `.parquet(path)` | Salva no formato Parquet (padrão do Spark, colunar, comprimido) |
| `.csv(path)` | Salva como CSV |
| `.json(path)` | Salva como JSON |
| `.saveAsTable(nome)` | Registra como tabela permanente no catálogo Hive/Spark |
| `.mode("overwrite")` | Sobrescreve se já existir |
| `.mode("append")` | Adiciona aos dados existentes |
| `.mode("ignore")` | Não faz nada se já existir |
| `.mode("error")` | Lança erro se já existir (padrão) |
| `.option("header", True)` | Inclui cabeçalho (CSV) |
| `.partitionBy("col")` | Particiona os arquivos pela coluna |

```python
df.write.mode("overwrite").parquet("/caminho/saida")
df.write.mode("overwrite").option("header", True).csv("/caminho/saida")
df.write.mode("overwrite").saveAsTable("esquema.tabela")
```

Salvar como Parquet — formato colunar padrão do Spark, eficiente para leitura posterior:

In [ ]:
df_trip.write.mode("overwrite").parquet("./trips_parquet")

Salvar como CSV com cabeçalho:

In [ ]:
df_trip.write.mode("overwrite").option("header", True).csv("./trips_csv")

Salvar como tabela permanente no catálogo Hive/Spark, adicionando uma coluna calculada:

In [ ]:
df_trip \
    .withColumn("tipo",
        when(col("duration") < 600,  "curta")
        .when(col("duration") < 3600, "média")
        .otherwise("longa")
    ) \
    .write.mode("overwrite").saveAsTable("bike.trip_classificada")

Verificar que a tabela foi criada e ler de volta para confirmar:

In [ ]:
spark.sql("SHOW TABLES IN bike").show()
spark.read.table("bike.trip_classificada").show(5)

**Exercício 8**: Crie uma função `ex08` que recebe `df_trip` e um inteiro `k` e retorna apenas as viagens feitas por assinantes (`subscription_type == "Subscriber"`) com duração entre 5 e 30 minutos (inclusive). O resultado deve ter as colunas `id`, `subscription_type`, `duration` e `start_station_name`.

**Obs**:
- Ordene de forma decrescente pela duração da viagem. Em caso de empate, ordene de forma crescente pelo `id` da viagem.
- Limite o resultado para `k` linhas

In [ ]:
def ex08(df_trip, k):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex08(df_trip, 15).toPandas()

In [ ]:
ia.sender(answer="ex08", task="spark_dataframes", question="ex08", answer_type="pycode")

**Exercício 9**: Crie uma função `ex09` que recebe `df_trip` e um inteiro `k` e retorna um DataFrame com as colunas `id`, `start_station_name`, `end_station_name`, `duration` e uma nova coluna `duration_hours` com a duração da viagem em horas, arredondada com 2 casas decimais.

**Obs**:
- Ordene de forma decrescente pelo `id`
- Limite o resultado para as `k` primeiras linhas.

In [ ]:
def ex09(df_trip, k):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex09(df_trip, 20).toPandas()

In [ ]:
ia.sender(answer="ex09", task="spark_dataframes", question="ex09", answer_type="pycode")

**Exercício 10**: Crie uma função `ex10` que recebe `df_trip` e retorna as 10 estações de partida (`start_station_name`) com mais viagens. O resultado deve ter as colunas `start_station_name` e `total_partidas`, ordenado de forma decrescente.

In [ ]:
def ex10(df_trip):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex10(df_trip).show(50)

In [ ]:
ia.sender(answer="ex10", task="spark_dataframes", question="ex10", answer_type="pycode")

**Exercício 11**: Crie uma função `ex11` que recebe `df_station` e retorna um DataFrame com as seguintes informações por cidade (`city`): `qtd_estacoes` (quantidade de estações), `media_docks` (média de `dock_count`, arredondada com 2 casas decimais), `max_docks` e `min_docks`.

**Obs**:
- Ordene o resultado de forma crescente por `qtd_estacoes` e depois de forma decrescente por `city`. 

In [ ]:
def ex11(df_station):
    # Seu código AQUI!
    return df_station  # Pode alterar o retorno, se necessário!

ex11(df_station).show()

In [ ]:
ia.sender(answer="ex11", task="spark_dataframes", question="ex11", answer_type="pycode")

**Exercício 12**: Crie uma função `ex12` que recebe `df_trip` e `df_station` e retorna, para cada cidade (`city`), a quantidade total de viagens que partiram de uma estação naquela cidade (`total_trips`). O resultado deve estar ordenado de forma decrescente.

*Dica*: o campo `start_station_id` em `df_trip` corresponde ao campo `id` em `df_station`.

In [ ]:
def ex12(df_trip, df_station):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex12(df_trip, df_station).show(50)

In [ ]:
ia.sender(answer="ex12", task="spark_dataframes", question="ex12", answer_type="pycode")

**Exercício 13**: Crie uma função `ex13` que recebe `df_trip` e retorna um DataFrame com as colunas `start_station_name`, `total_viagens` e `categoria`, onde `categoria` classifica cada estação de partida pelo seu volume de viagens como:
- `"movimentada"`: mais de 10000 viagens
- `"moderada"`: entre 1000 e 10000 viagens (inclusive)
- `"tranquila"`: menos de 1000 viagens

**Obs**:
- O resultado deve estar **ordenado de forma crescente por `start_station_name`**.
- Limite o resultado para as 20 primeiras linhas.

*Dica*: agrupe primeiro, depois use `when` sobre a coluna calculada.

In [ ]:
def ex13(df_trip):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex13(df_trip).show(50)

In [ ]:
ia.sender(answer="ex13", task="spark_dataframes", question="ex13", answer_type="pycode")

**Exercício 14**: Crie uma função `ex14` que recebe `df_trip` e retorna, para cada `subscription_type`, as colunas `total_viagens`, `viagens_longas` (duração > 1 hora) e `pct_longas` (percentual de viagens longas em relação ao total, arredondado com 2 casas decimais).

*Dica*: use `when` para criar uma flag binária antes de agregar com `sum`.

In [ ]:
def ex14(df_trip):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex14(df_trip).show()

In [ ]:
ia.sender(answer="ex14", task="spark_dataframes", question="ex14", answer_type="pycode")

**Exercício 15**: Crie uma função `ex15` que recebe `df_trip` e retorna as 5 bicicletas (`bike_id`) mais utilizadas, com as colunas `bike_id`, `total_viagens` e `total_horas` (soma de todas as durações em horas, arredondada com 2 casas decimais). Ordene pelo número de viagens de forma decrescente.

In [ ]:
def ex15(df_trip):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex15(df_trip).show(50)

In [ ]:
ia.sender(answer="ex15", task="spark_dataframes", question="ex15", answer_type="pycode")

**Exercício 16**: Crie uma função `ex16` que recebe `df_trip` e `df_station` e retorna, para cada cidade (`city`), a **duração média das viagens em minutos** (`duracao_media_min`, arredondada com 2 casas decimais), considerando **apenas viagens feitas por clientes sem assinatura** (`subscription_type == "Customer"`). Ordene pelo resultado de forma decrescente.

In [ ]:
def ex16(df_trip, df_station):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

ex16(df_trip, df_station).show(50)

In [ ]:
ia.sender(answer="ex16", task="spark_dataframes", question="ex16", answer_type="pycode")

**Desafio**: Crie uma função `desafio01` que recebe `df_trip` e retorna as 10 estações de partida (`start_station_name`) com a maior **proporção de viagens circulares**, ou seja, aquelas em que a estação de partida é a mesma que a de chegada (`start_station_id == end_station_id`).

**Obs**:
- Considere apenas estações com pelo menos 10 viagens.
- O DataFrame deve ter as colunas `start_station_name`, `total_trips`, `round_trips` e `round_trip_ratio` (proporção arredondada com 4 casas decimais), ordenado por `round_trip_ratio` de forma decrescente.

In [ ]:
def desafio01(df_trip):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

desafio01(df_trip).show(50)

In [ ]:
ia.sender(answer="ex_desafio01", task="spark_dataframes", question="ex_desafio01", answer_type="pycode")

**Desafio 2**: Crie uma função `desafio02` que recebe `df_trip` e `df_station` e retorna as **10 rotas mais populares**, ou seja, pares `(start_station_name, end_station_name)`, excluindo viagens circulares (`start_station_id != end_station_id`).

O resultado deve conter as colunas:
- `start_station_name` — nome da estação de partida
- `end_station_name` — nome da estação de chegada
- `start_city` — cidade da estação de partida (obtida via join com `df_station`)
- `total_trips` — número de viagens nessa rota
- `avg_duration_min` — duração média em **minutos**, arredondada com 2 casas decimais

Considere apenas rotas com **pelo menos 10 viagens**. Ordene por `total_trips` decrescente.

In [ ]:
def desafio02(df_trip, df_station):
    # Seu código AQUI!
    return df_trip  # Pode alterar o retorno, se necessário!

desafio02(df_trip, df_station).show(50, truncate=False)

In [ ]:
ia.sender(answer="ex_desafio02", task="spark_dataframes", question="ex_desafio02", answer_type="pycode")

## Conferir Notas

In [ ]:
ia.tasks()

In [ ]:
ia.grades(task="spark_dataframes")

In [ ]:
ia.grades(by="TASK", task="spark_dataframes")

In [ ]:
# Calcular média de ATVs, desconsiderando duas atividades (divide por n-2)
ia.average(excluded_count=2)